# Generador de Ruido (Dataset ALDIMI)
Este notebook toma el dataset original `cancer patient data sets.xlsx` y aplica técnicas de **Data Augmentation con Ruido** (Características y Etiquetas). 
El objetivo es dificultar el aprendizaje para los modelos, pasando de un problema determinista (F1=1.0) a uno probabilístico realista.

In [1]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)

## 1. Carga del Dataset Original

In [2]:
# Cargar datos
df_original = pd.read_excel('../cancer patient data sets.xlsx')

if 'Patient Id' in df_original.columns:
    df_original = df_original.drop(['Patient Id'], axis=1)
if 'index' in df_original.columns:
    df_original = df_original.drop(['index'], axis=1)

print(f"Dimensiones originales: {df_original.shape}")
display(df_original.head())

Dimensiones originales: (1000, 24)


,Age,Gender,Air Pollution,Alcohol use,Dust Allergy,OccuPational Hazards,Genetic Risk,chronic Lung Disease,Balanced Diet,Obesity,...,Fatigue,Weight Loss,Shortness of Breath,Wheezing,Swallowing Difficulty,Clubbing of Finger Nails,Frequent Cold,Dry Cough,Snoring,Level
0,33,1,2,4,5,4,3,2,2,4,...,3,4,2,2,3,1,2,3,4,Low
1,17,1,3,1,5,3,4,2,2,2,...,1,3,7,8,6,2,1,7,2,Medium
2,35,1,4,5,6,5,5,4,6,7,...,8,7,9,2,1,4,6,7,2,High
3,37,1,7,7,7,7,6,7,7,7,...,4,2,3,1,4,5,6,7,5,High
4,46,1,6,8,7,7,7,6,7,7,...,3,2,4,1,4,2,4,2,3,High


## 2. Definición de Funciones de Ruido
Inyectaremos perturbaciones (±1 o ±2) en variables numéricas ordinales y errores de etiqueta (5%).

In [3]:
def inject_feature_noise(df, noise_prob=0.3):
    df_noisy = df.copy()
    columns_to_perturb = [c for c in df_noisy.columns if c not in ['Age', 'Gender', 'Level']]
    
    for col in columns_to_perturb:
        mask = np.random.rand(len(df_noisy)) < noise_prob
        noise = np.random.choice([-1, 1], size=len(df_noisy))
        df_noisy.loc[mask, col] = df_noisy.loc[mask, col] + noise[mask]
        
        df_noisy[col] = df_noisy[col].clip(1, 9)
        
    age_mask = np.random.rand(len(df_noisy)) < noise_prob
    age_noise = np.random.randint(-5, 6, size=len(df_noisy))
    df_noisy.loc[age_mask, 'Age'] = df_noisy.loc[age_mask, 'Age'] + age_noise[age_mask]
    df_noisy['Age'] = df_noisy['Age'].clip(10, 90) 
    
    return df_noisy

def inject_label_noise(df, flip_prob=0.05):
    df_noisy = df.copy()
    levels = ['Low', 'Medium', 'High']
    mask = np.random.rand(len(df_noisy)) < flip_prob
    
    random_labels = np.random.choice(levels, size=mask.sum())
    df_noisy.loc[mask, 'Level'] = random_labels
    return df_noisy

## 3. Generación del Dataset Ampliado (Data Augmentation)
Crearemos 2 versiones ruidosas por cada fila original, multiplicando el tamaño x3.

In [4]:
print("Generando Copia Ruidosa 1...")
df_noisy_1 = inject_feature_noise(df_original, noise_prob=0.3)
df_noisy_1 = inject_label_noise(df_noisy_1, flip_prob=0.05)

print("Generando Copia Ruidosa 2...")
df_noisy_2 = inject_feature_noise(df_original, noise_prob=0.4) 
df_noisy_2 = inject_label_noise(df_noisy_2, flip_prob=0.08)

df_augmented = pd.concat([df_original, df_noisy_1, df_noisy_2], ignore_index=True)

df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dimensiones del dataset ampliado: {df_augmented.shape}")

Generando Copia Ruidosa 1...
Generando Copia Ruidosa 2...
Dimensiones del dataset ampliado: (3000, 24)


## 4. Exportación

In [5]:
out_dir = '../datos/datos_modelo2/'
os.makedirs(out_dir, exist_ok=True)

out_file = os.path.join(out_dir, 'cancer_patient_data_sets_noisy.csv')
df_augmented.to_csv(out_file, index=False)

print(f"✅ Dataset ruidoso guardado en: {out_file}")

✅ Dataset ruidoso guardado en: ../datos/datos_modelo2/cancer_patient_data_sets_noisy.csv
